# Prompt & Agent Optimization with Opik — an A-to-Z Guide

This notebook takes you from *"I have a prompt that works okay"* to *"my prompt and my agent are measurably better, and every improvement is a comparable run in Opik."*

**Two ways to use it:**
- **Live workshop (≈20 min):** run **Part 1** top to bottom. You'll optimize a RAG answer prompt against an exact-match metric and see the improvement in Opik.
- **Take-home guide:** continue through Parts 2–5 — LLM-judge metrics (and how to *trust* them), multi-objective optimization, and optimizing an agent's tool use.

We optimize a **documentation assistant** for a fictional product, **Ledgerline** (a task-queue API), so the corpus is clean and the lesson is about *optimization*, not about parsing messy docs.

## Part 0 — How to think about prompt optimization

**When do you start?** When you have (1) a prompt that works *okay*, (2) a dataset of representative inputs, and (3) a metric that says how good an output is — and hand-tuning has plateaued.

**The mental shift.** Classic optimization gives you an objective and a gradient. Prompt optimization is different: the **search space is prompt text**, and the **objective is a metric computed over a dataset**. You can't differentiate it, so optimizers *propose* candidate prompts, *evaluate* them on your dataset, keep the best, and repeat.

The three ingredients map exactly to three objects you'll build:

| Ingredient | Opik object |
|---|---|
| The prompt | `ChatPrompt` |
| The dataset | Opik `Dataset` |
| The metric | a callable `(dataset_item, llm_output) -> float` |

The loop, once, looks like: **propose candidate → evaluate on dataset → keep best → repeat.** Everything below is that loop, escalating in complexity.

In [ ]:
import opik
from optimization_guide import config, data, rag_app

# Fail fast with a clear message if credentials are missing.
config.check_prerequisites()

client = opik.Opik(project_name=config.PROJECT_NAME)
print("Using models:", config.GEN_MODEL)

### Part 1 — Your first optimization ⭐ (workshop)

We'll ingest the Ledgerline docs, build an evaluation dataset with **checkable answers**, score a baseline prompt with an **exact-match metric** (no LLM judge needed), then let an optimizer improve the prompt.

In [ ]:
docs = data.load_docs()
count = rag_app.ingest(docs)
print(f"Ingested {count} doc snippets into ChromaDB")

In [ ]:
exact_cases = data.load_exact_cases()
exact_dataset = data.build_dataset(client, "ledgerline-exact", exact_cases)
print(f"Dataset 'ledgerline-exact' has {len(exact_cases)} cases")

#### The metric: exact-match, no judge

Our first metric is deterministic and cheap: **does the answer contain the expected fact?** Opik ships `Contains` for exactly this. Optimizer metrics are plain callables `(dataset_item, llm_output) -> float`, so we wrap `Contains` in one. *Not every metric needs an LLM.*

In [ ]:
from opik.evaluation.metrics import Contains


def exact_match(dataset_item: dict, llm_output: str) -> float:
    # Contains returns 1.0 if expected_substring is in the output, else 0.0.
    result = Contains(case_sensitive=False).score(
        output=llm_output,
        reference=dataset_item["expected_substring"],
    )
    return result.value


exact_match.__name__ = "exact_match"

#### The starting prompt

Here is our baseline system prompt — deliberately mediocre, so there's room to improve. This is the `ChatPrompt` the optimizer will rewrite. `{query}` is filled from each dataset row.

In [ ]:
from opik_optimizer import ChatPrompt

BASELINE_SYSTEM = "You are a support bot. Answer the question."

prompt = ChatPrompt(
    name="ledgerline-answer",
    system=BASELINE_SYSTEM,
    user="{query}",
    model=config.GEN_MODEL,
)

#### Run the optimizer

We use **`MetaPromptOptimizer`** — it uses a reasoning LLM to critique and rewrite the prompt. It's the docs' recommended general-purpose starting point for prompt wording. Watch the params:
- `max_trials` — how many candidate prompts to try.
- `n_samples` — dataset rows evaluated per candidate (smaller = cheaper/faster for a live run).
- `skip_perfect_score=False` — keep optimizing even if the baseline already scores high.

In [ ]:
from opik_optimizer import MetaPromptOptimizer

optimizer = MetaPromptOptimizer(
    model=config.OPTIMIZER_MODEL,
    n_threads=4,
    skip_perfect_score=False,
)

result = optimizer.optimize_prompt(
    prompt=prompt,
    dataset=exact_dataset,
    metric=exact_match,
    max_trials=8,
    n_samples=8,
)

print("Baseline score:", result.initial_score)
print("Best score:    ", result.score)
print("\nOptimized system prompt:\n", result.prompt)

#### See it in Opik

Open **Evaluation → Optimization runs** in your Opik workspace. You'll see this run with every candidate prompt, its score, and the trace for each trial. Compare the baseline row to the best row — that delta is your improvement.

🎓 **This is where the live workshop ends.** You've run a real optimization and improved a prompt, measured against a dataset, stored in Opik. Everything below builds on exactly this loop.

## Part 2 — Metrics done right

Exact-match got us far because our questions had crisp answers. But real docs questions are open-ended — *"How should I handle a job that keeps failing?"* has no single substring. For those you need a metric that judges **meaning**: an **LLM-as-judge**.

#### The LLM-judge metric

Opik ships judge metrics like `AnswerRelevance` (is the answer relevant to the question, given context?) and `Hallucination` (is it unsupported by context?). We wrap `AnswerRelevance` as an optimizer metric, exactly like we wrapped `Contains` — same callable shape, different scorer.

In [ ]:
from opik.evaluation.metrics import AnswerRelevance


def answer_relevance(dataset_item: dict, llm_output: str) -> float:
    result = AnswerRelevance(model=config.JUDGE_MODEL).score(
        input=dataset_item["query"],
        output=llm_output,
        context=[dataset_item["reference"]],
    )
    return result.value


answer_relevance.__name__ = "answer_relevance"

#### How do we *trust* a judge?

An LLM-judge is itself a prompt — it can be wrong. Before you optimize *against* it, sanity-check it:

1. **Spot-check against your own labels.** Take 3–5 rows, decide the score yourself, and compare. If you and the judge disagree wildly, fix the judge before trusting its numbers.
2. **Read the *reason*, not just the number.** Opik judge metrics return a `reason`. A right score for the wrong reason is a red flag.
3. **Watch for drift and bias.** Judges favor longer, confident-sounding answers. If your metric rewards verbosity, your "optimized" prompt may just be wordier — which is exactly why Part 2 ends with a *cost/length* objective.

Run the cell below to inspect a judge score **and its reasoning** on one example.

In [ ]:
sample = data.load_judge_cases()[0]
sample_output = rag_app.answer(sample["query"], system_prompt=result.prompt.system)  # result.prompt is a ChatPrompt; .system is the optimized system text
judged = AnswerRelevance(model=config.JUDGE_MODEL).score(
    input=sample["query"],
    output=sample_output,
    context=[sample["reference"]],
)
print("Question:", sample["query"])
print("Answer:  ", sample_output)
print("Score:   ", judged.value)
print("Reason:  ", judged.reason)

In [ ]:
judge_cases = data.load_judge_cases()
judge_dataset = data.build_dataset(client, "ledgerline-judge", judge_cases)

judge_prompt = ChatPrompt(
    name="ledgerline-answer-judge",
    system=BASELINE_SYSTEM,
    user="{query}",
    model=config.GEN_MODEL,
)

judge_result = optimizer.optimize_prompt(
    prompt=judge_prompt,
    dataset=judge_dataset,
    metric=answer_relevance,
    max_trials=8,
    n_samples=8,
)
print("Judge-metric baseline:", judge_result.initial_score, "-> best:", judge_result.score)

#### Multi-objective: quality *and* cost

Optimizing purely for a judge can inflate answer length. Often you want **quality high *and* answers short**. `MultiMetricObjective` combines metrics into one weighted composite the optimizer maximizes — this is how Opik does multi-objective optimization.

Below we combine `answer_relevance` (weight 0.7) with a length penalty (weight 0.3). The length metric is a plain callable that returns a normalized "shorter is better" score.

In [ ]:
from opik_optimizer import MultiMetricObjective


def brevity(dataset_item: dict, llm_output: str) -> float:
    # Normalized "shorter is better": 1.0 for <=200 chars, decaying to 0 at 1000 chars.
    length = len(llm_output)
    return max(0.0, min(1.0, (1000 - length) / 800))


brevity.__name__ = "brevity"

composite = MultiMetricObjective(
    metrics=[answer_relevance, brevity],
    weights=[0.7, 0.3],
    name="relevance_and_brevity",
)

multi_result = optimizer.optimize_prompt(
    prompt=judge_prompt,
    dataset=judge_dataset,
    metric=composite,
    max_trials=8,
    n_samples=8,
)
print("Multi-objective best score:", multi_result.score)
print("\nOptimized prompt:\n", multi_result.prompt)

#### Compare your runs

You now have three optimization runs in Opik: exact-match, judge, and multi-objective. In **Evaluation → Optimization runs**, put them side by side. Notice how the multi-objective prompt trades a little relevance for much shorter answers — that trade-off is the whole point of naming your objectives explicitly.

## Part 3 — From prompt to agent

So far we optimized a single answer prompt. Real systems are agents: they *decide* what to do. Our RAG app can grow a **retrieval gate** — decide whether a question even needs a docs lookup (cheap questions skip retrieval). That decision is itself a prompt, and the same optimizer loop tunes it.

In [ ]:
@opik.track
def agentic_answer(query: str, system_prompt: str) -> str:
    # Agent step: decide whether to retrieve, then answer accordingly.
    if rag_app.should_retrieve(query):
        return rag_app.answer(query, system_prompt=system_prompt)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]
    import litellm
    resp = litellm.completion(model=config.GEN_MODEL, messages=messages)
    return resp.choices[0].message.content

#### When demonstrations matter: Few-Shot Bayesian

If the win comes from *showing examples* rather than rewording instructions, reach for `FewShotBayesianOptimizer` — it uses Bayesian search (Optuna) to pick the best set and order of few-shot demonstrations to attach to your prompt. Same API as before.

In [ ]:
from opik_optimizer import FewShotBayesianOptimizer

fewshot_optimizer = FewShotBayesianOptimizer(model=config.OPTIMIZER_MODEL, n_threads=4)

fewshot_result = fewshot_optimizer.optimize_prompt(
    prompt=judge_prompt,
    dataset=judge_dataset,
    metric=answer_relevance,
    n_samples=8,
)
print("Few-shot best score:", fewshot_result.score)

#### Tuning the model, not the prompt: Parameter optimizer

Sometimes the prompt is fine and you just need better sampling settings. `ParameterOptimizer` leaves the prompt alone and searches temperature / top_p with Bayesian optimization. It's the right reach when behavior — not wording — is the problem. See the [Parameter optimizer docs](https://www.comet.com/docs/opik/agent_optimization/algorithms/parameter_optimizer) for the search-space API.

## Part 4 — Choosing an optimizer

You've now *used* several optimizers at the moment each was the right tool. Here's the consolidated map:

| Optimizer | Best for | You saw it in |
|---|---|---|
| **MetaPrompt** | General prompt rewording & clarity | Part 1 |
| **HRPO** | Systematic fixes from *why* prompts fail (failure-mode analysis) | (try on your own) |
| **Few-Shot Bayesian** | Picking the best demonstrations | Part 3 |
| **Evolutionary** | Exploring diverse structures; multi-objective | (see multi-objective, Part 2) |
| **GEPA** | Single-turn, reflection-heavy tasks (`pip install gepa`) | (try on your own) |
| **Parameter** | Temperature / top_p, prompt unchanged | Part 3 |

**How to choose, in four questions:**
1. **What's the constraint** — wording, examples, tool use, or sampling params?
2. **Is the dataset ready** — reflective optimizers (HRPO) need metrics with detailed *reasons*. Split train/validation to avoid overfitting.
3. **What's the budget** — Evolutionary/GEPA burn more tokens than MetaPrompt.
4. **Can you chain?** — e.g. MetaPrompt to fix wording, then Parameter to tune sampling.

The docs' own advice: **start with GEPA or HRPO** for a new task, then specialize.

#### Chaining optimizers

Because every optimizer shares the same API and returns an `OptimizationResult` whose `.prompt` you can feed into the next, you can chain them: optimize wording, then feed the winner into a Parameter run. See [Chaining optimizers](https://www.comet.com/docs/opik/agent_optimization/advanced/chaining_optimizers).

## Part 5 — Take it further

**Version the winner.** Promote your best prompt to the Opik **Prompt Library** so it's versioned and reusable.

In [ ]:
# multi_result.prompt is a ChatPrompt; .system is the optimized system text
best_prompt = opik.Prompt(name="ledgerline-answer", prompt=multi_result.prompt.system)
print("Saved prompt version:", best_prompt.commit)

**Where to go next:**
- **[Optimization Studio](https://www.comet.com/docs/opik/agent_optimization/optimization_studio)** — run all of this from the Opik UI, no code.
- **[Optimizer benchmarks](https://www.comet.com/docs/opik/agent_optimization/algorithms/benchmarks)** — numbers per algorithm.
- **[Agent optimization overview](https://www.comet.com/docs/opik/agent_optimization/overview)** — the full reference.
- **Wrap this in a CLI** — the plumbing (`optimization_guide/`) is import-ready; turning the notebook into a repeatable CLI is a natural next project (out of scope here).

You've gone A-to-Z: framing → first optimization → trustworthy judge metrics → multi-objective → agent tuning → optimizer selection → versioned prompt. Every step is a comparable run in Opik.